# 08 — VOSviewer Preparation

Genera tesauros auditables, listas de selección, redes de keywords y una guía exacta para VOSviewer.

> Notebook reproducible del Paper Semantic Bibliometric Toolkit v1.0.0. No elimina documentos ni convierte resultados exploratorios en conclusiones sin revisión manual.

## 1. Cargar frecuencias y reglas

In [ ]:

from pathlib import Path
import sys
import json
import pandas as pd
from IPython.display import display

PROJECT_ROOT = None
current = Path.cwd().resolve()
for candidate in [current] + list(current.parents):
    if (candidate / "config.json").exists():
        PROJECT_ROOT = candidate
        break
if PROJECT_ROOT is None:
    raise FileNotFoundError("No se encontró config.json.")

sys.path.insert(0, str(PROJECT_ROOT / "src"))

from core import (
    find_project_root, read_json, load_config, ensure_directories,
    load_scopus_csv, load_clean_corpus, normalize_spacing,
    normalize_identifier, normalize_title, normalize_keyword,
    split_semicolon, frequency_table, save_csv, save_json,
    markdown_table, combined_text
)

CONFIG = load_config(PROJECT_ROOT)
ensure_directories(PROJECT_ROOT)
DATA_DIR = PROJECT_ROOT / "data"
OUTPUT_DIR = PROJECT_ROOT / "outputs"
FIGURES_DIR = PROJECT_ROOT / "figures"
REPORTS_DIR = PROJECT_ROOT / "reports"
THESAURUS_DIR = PROJECT_ROOT / "thesaurus"
VALIDATION_DIR = PROJECT_ROOT / "validation"

print("Project root:", PROJECT_ROOT)
print("Toolkit version:", CONFIG.get("toolkit_version"))
print("Configured corpus:", PROJECT_ROOT / CONFIG["input_csv"])

frequency = pd.read_csv(OUTPUT_DIR / "03_author_keyword_frequency.csv", encoding="utf-8-sig")
rules = read_json(PROJECT_ROOT / "config" / "keyword_rules.json")


## 2. Tesauro base y versión sin términos de búsqueda

In [ ]:

merge_rows = [{"label": left, "replace by": right, "Action": "merge", "Reason": "Conservative normalization"} for left, right in rules.get("known_merges", {}).items()]
exclude_rows = [{"label": term, "replace by": "", "Action": "exclude candidate", "Reason": "Generic term - manual audit"} for term in rules.get("generic_exclusion_candidates", [])]
search_rows = [{"label": term, "replace by": "", "Action": "optional search-term removal", "Reason": "Term forced by query"} for term in rules.get("search_terms_for_optional_removal", [])]
base_audit = pd.DataFrame(merge_rows + exclude_rows).drop_duplicates("label")
without_search_audit = pd.DataFrame(merge_rows + exclude_rows + search_rows).drop_duplicates("label")
save_csv(base_audit, THESAURUS_DIR / "thesaurus_base_audit.csv")
save_csv(without_search_audit, THESAURUS_DIR / "thesaurus_without_search_terms_audit.csv")
base_audit[["label", "replace by"]].to_csv(THESAURUS_DIR / "thesaurus_base.txt", index=False, encoding="utf-8-sig")
without_search_audit[["label", "replace by"]].to_csv(THESAURUS_DIR / "thesaurus_without_search_terms.txt", index=False, encoding="utf-8-sig")
display(base_audit)


## 3. Listas de keywords para umbrales 2–5

In [ ]:

for threshold in CONFIG.get("keyword_thresholds_to_export", [2, 3, 4, 5]):
    selected = frequency[frequency["Documents"] >= int(threshold)].copy()
    selected["Threshold"] = int(threshold)
    selected["Generic exclusion candidate"] = selected["Keyword normalized"].isin(rules.get("generic_exclusion_candidates", []))
    selected["Search term"] = selected["Keyword normalized"].isin(rules.get("search_terms_for_optional_removal", []))
    save_csv(selected, THESAURUS_DIR / ("keyword_selection_min" + str(threshold) + ".csv"))
    print("Threshold", threshold, "->", len(selected), "keywords")


## 4. Comprobación del RIS y guía VOSviewer

In [ ]:

ris_path = PROJECT_ROOT / CONFIG.get("input_ris", "data/scopus_export.ris")
ris_status = pd.DataFrame([{"RIS path": str(ris_path), "Exists": ris_path.exists(), "Action": "Ready" if ris_path.exists() else "Copy the complete Scopus RIS to this path"}])
save_csv(ris_status, OUTPUT_DIR / "08_ris_status.csv")
display(ris_status)

guide = """
VOSviewer protocol
==================
1. Create a map based on bibliographic data.
2. Read data from reference-manager files and load: {ris_path}
3. Type of analysis: Co-occurrence.
4. Unit of analysis: Author keywords.
5. Counting method: Fractional counting.
6. Test occurrence thresholds 3 and 4 first.
7. Load thesaurus/thesaurus_base.txt.
8. Generate Network, Overlay (average publication year), and Density visualizations.
9. Repeat with thesaurus_without_search_terms.txt to reveal technologies and tasks without query-forced terms.
10. Record selected keywords, clusters, and all parameter values.
11. Export vector files when possible; Illustrator may improve typography but must not alter scientific links.
""".format(ris_path=ris_path)
(THESAURUS_DIR / "VOSviewer_protocol.txt").write_text(guide.strip(), encoding="utf-8")
print(guide)


## 5. Protocolo metodológico

In [ ]:

text = (
    "A conservative thesaurus was generated from explicit lexical rules. A second version removed terms directly forced by the "
    "Scopus query, enabling an auxiliary visualization focused on technologies and tasks. Both versions retain a complete audit trail."
)
(REPORTS_DIR / "08_vosviewer_methodology.txt").write_text(text, encoding="utf-8")
print(text)
